# Debugging Demo

## Within a notebook

In [ ]:
def factorial(n):  # Recursive factorial function
    return 1 if n == 0 else n * factorial(n - 1)


def generate_factorials(start, end):  # Inclusive range
    factorials = []  # store factorials
    for i in range(start, end + 1):
        factorials.append(factorial(i))
    return factorials


def sum_factorials(factorials):  # Sum a list
    total = 0
    for f in factorials:
        total += f
    return total


ranges = [(0, 3), (4, 7), (8, 10)]  # List of ranges
factorial_results = {}  # Results for each range

for start, end in ranges:  # Calc factorials for each range
    factorial_list = generate_factorials(start, end)
    factorial_results[(start, end)] = factorial_list

factorial_sums = {}  # Sum for each range
for r, factorial_list in factorial_results.items():
    factorial_sums[r] = sum_factorials(factorial_list)

# Trigger an error to enter debug mode
print("Factorials and their sums for each range:")
print(factorial_results)
print(factorial_sums)
print("This will cause KeyError:", factorial_sums[(11, 15)])
print("Completed calculating factorials and their sums.")

## Within a schema

This will import the local `schema_template`, insert some
dummy subjects, and try po run an analysis on each.

In [ ]:
import os
from pathlib import Path
import datajoint as dj

path_name = Path(".").resolve().name
if Path(".").resolve().name != "spyglass_custom":
    print("moving")
    os.chdir("../../..")

dj.config.load("dj_local_conf.json")
dj.conn()  # Note to self: Docker

In [ ]:
from spyglass.common import Subject
from spyglass_custom.cbroz1 import schema_template as st

key = Subject().fetch(as_dict=True)[0]
_ = key.pop("subject_id")
for new_id in [f"{c}_{n}" for c in "ab" for n in range(1, 5)]:
    Subject().insert1(dict(**key, subject_id=new_id), skip_duplicates=True)

Subject()
st.SubjBlinded().blind_subjects(restriction="subject_id LIKE 'b%'")

st.MyAnalysisSelection().insert_all()
st.MyAnalysis().delete(safemode=False)

st.MyAnalysis().populate()  # Must place breakpoint on problem before debug